# TP - Partie 1 : Arbre de Decision from scratch

Implementer un arbre de decision binaire vous-même (sans sklearn).

## Pseudo-code : Gini / Entropie
```py
FONCTION gini(y):
    compter les occurrences de chaque classe
    calculer les proportions p_i = count_i / total
    RETOURNER 1 - somme(p_i^2)

FONCTION entropy(y):
    compter les occurrences de chaque classe
    calculer les proportions p_i = count_i / total
    RETOURNER -somme(p_i * log2(p_i))  # ignorer si p_i = 0
```

## Pseudo-code : Trouver le meilleur split
```py
FONCTION find_best_split(X, y):
    best_gain = 0
    POUR chaque feature f:
        POUR chaque valeur unique v dans X[:, f]:
            separer en left (X[f] <= v) et right (X[f] > v)
            SI left ou right est vide: continuer
            calculer impurete_left et impurete_right
            gain = impurete_parent - moyenne_ponderee(impuretes)
            SI gain > best_gain: sauvegarder f, v, gain
    RETOURNER best_feature, best_threshold, best_gain
```

## Pseudo-code : Construction recursive
```py
FONCTION build_tree(X, y, depth):
    SI depth >= max_depth OU une seule classe OU pas assez d'echantillons:
        RETOURNER feuille avec classe majoritaire
    
    trouver le meilleur split
    SI pas de gain possible:
        RETOURNER feuille avec classe majoritaire
    
    separer les donnees selon le split
    left_subtree = build_tree(X_left, y_left, depth+1)
    right_subtree = build_tree(X_right, y_right, depth+1)
    RETOURNER noeud(feature, threshold, left, right)
```

## Pseudo-code : Prediction
```py
FONCTION predict_sample(node, x):
    SI node est une feuille:
        RETOURNER node.predicted_class
    SI x[node.feature] <= node.threshold:
        RETOURNER predict_sample(node.left, x)
    SINON:
        RETOURNER predict_sample(node.right, x)
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree


## 1. Chargement des donnees


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
           'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(url, names=columns, na_values='?')
df = df.dropna()
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1).values
y = df['target'].values
feature_names = df.drop('target', axis=1).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Echantillons : {len(df)}")
print(f"Features     : {len(feature_names)}")


## 2. Classe DecisionTreeNode


In [ ]:
class DecisionTreeNode:
    """Represente un noeud de l'arbre de decision (interne ou feuille)."""
    
    def __init__(self, feature=None, threshold=None, left=None, right=None, 
                 predicted_class=None, n_samples=0, class_counts=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.predicted_class = predicted_class
        self.n_samples = n_samples
        self.class_counts = class_counts
    
    def is_leaf(self):
        return self.predicted_class is not None


## 3. Calcul de l'impurete


In [ ]:
def gini_impurity(y):
    """Calcule l'indice de Gini : Gini(S) = 1 - sum(p_i^2)"""
    if len(y) == 0:
        return 0
    # TODO
    pass


def entropy(y):
    """Calcule l'entropie : H(S) = -sum(p_i * log2(p_i))"""
    if len(y) == 0:
        return 0
    # TODO
    pass


## 4. Trouver le meilleur split


In [ ]:
def find_best_split(X, y, criterion='gini'):
    """Trouve la meilleure feature et seuil pour separer les donnees."""
    n_samples, n_features = X.shape
    impurity_func = gini_impurity if criterion == 'gini' else entropy
    parent_impurity = impurity_func(y)
    
    best_gain = 0
    best_feature = None
    best_threshold = None
    
    # TODO
    
    return best_feature, best_threshold, best_gain


## 5. Construction recursive de l'arbre


In [ ]:
def build_tree(X, y, depth=0, max_depth=5, min_samples_split=2, criterion='gini'):
    """Construit recursivement l'arbre de decision."""
    n_samples = len(y)
    n_classes = len(np.unique(y))
    class_counts = [np.sum(y == c) for c in range(2)]
    majority_class = np.argmax(class_counts)
    
    # TODO
    pass


## 6. Prediction


In [ ]:
def predict_sample(node, x):
    """Predit la classe pour UN echantillon."""
    # TODO
    pass


def predict(node, X):
    """Predit les classes pour plusieurs echantillons."""
    return np.array([predict_sample(node, x) for x in X])


## 7. Visualisation


In [ ]:
def print_tree(node, feature_names, depth=0, prefix=""):
    """Affiche l'arbre de maniere textuelle."""
    indent = "    " * depth
    if node.is_leaf():
        class_name = "Malade" if node.predicted_class == 1 else "Sain"
        print(f"{indent}{prefix}[{class_name}] (n={node.n_samples})")
    else:
        feature_name = feature_names[node.feature]
        print(f"{indent}{prefix}{feature_name} <= {node.threshold:.2f} ? (n={node.n_samples})")
        print_tree(node.left, feature_names, depth + 1, "OUI -> ")
        print_tree(node.right, feature_names, depth + 1, "NON -> ")


## 8. Test


In [ ]:
my_tree = build_tree(X_train, y_train, max_depth=3, criterion='gini')

y_pred_my = predict(my_tree, X_test)
acc_my = accuracy_score(y_test, y_pred_my)

print(f"Mon arbre (max_depth=3) : {acc_my:.4f}")
print("\nStructure :")
print_tree(my_tree, feature_names)


## 9. Comparaison avec sklearn


In [ ]:
sklearn_tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
sklearn_tree.fit(X_train, y_train)
acc_sklearn = sklearn_tree.score(X_test, y_test)

print(f"Mon arbre : {acc_my:.4f}")
print(f"Sklearn   : {acc_sklearn:.4f}")

plt.figure(figsize=(16, 8))
plot_tree(sklearn_tree, feature_names=feature_names, class_names=['Sain', 'Malade'],
          filled=True, rounded=True, fontsize=10)
plt.title('Arbre Sklearn (pour comparaison)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
